# House Image Discovery with Cost Tracking

Finds the best panoramic views of **one target house** and describes it:

1. query pano metadata within `SEARCH_RADIUS_M` of the house (parameterised, dry-run first);
2. for every pano, pick in code the ground camera that faces the house and rank panos by
   distance and viewing angle;
3. render a perspective crop centred on the house bearing, sized to the house, and ask Gemini
   for a `HouseView` (visibility, occlusion, stories, material, roof type);
4. give the house a **geometry-anchored id** (hash of class + location), so repeated runs and
   other notebooks refer to the same house.

**Data model.** Each pano is a rosette of 7 frames (`observation_id` suffix `_0`..`_6`):
cameras 0-5 look horizontally ~60 deg apart, camera 6 looks at the sky. Frames are portrait
3648x5472 wide-angle images with a per-camera `camera_pose` (position, heading, pitch, roll)
and no published intrinsics; `svi_geo` ships a calibrated fisheye model for them. Panos are
chained into drive sequences from `snapshot_id` + capture time + distance (`capture_id` is one
id per pano and cannot be used for this); the spacing is measured, not assumed.

**Division of labour.** All geometry is code: lens undistortion, camera selection, view
rendering, box -> bearing conversion, triangulation, deduplication and smoothing. Gemini only
answers perception questions about images that code has already rendered, and every answer is
validated against a Pydantic schema.

In [ ]:
# Install the shared `svi_geo` helper package (geometry, sequencing, dedup, Gemini client).
# In a clone of the repo the local copy is used: $SVI_GEO_LOCAL_PATH, or the first parent
# directory that contains street_view_insights/panoramic/svi_geo. Otherwise pip installs it
# from GitHub at SVI_GEO_REF.
import importlib.util
import os
import pathlib
import subprocess
import sys

# TODO(after-merge): set to "main" once svi_geo is merged. Override with $SVI_GEO_REF
# (a commit SHA gives reproducible runs).
SVI_GEO_REF = os.environ.get("SVI_GEO_REF", "svi-gemini-improvements")


def find_local_svi_geo():
    if os.environ.get("SVI_GEO_LOCAL_PATH"):
        return os.environ["SVI_GEO_LOCAL_PATH"]
    here = pathlib.Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "street_view_insights/panoramic/svi_geo" / "pyproject.toml").is_file():
            return str(d / "street_view_insights/panoramic/svi_geo")
    return None


LOCAL_SVI_GEO = find_local_svi_geo()
if importlib.util.find_spec("svi_geo") is None:
    if LOCAL_SVI_GEO:
        src = f"{LOCAL_SVI_GEO}[notebooks]"
    else:
        src = (
            "svi-geo[notebooks] @ git+https://github.com/googlemaps-samples/insights-samples"
            f"@{SVI_GEO_REF}#subdirectory=street_view_insights/panoramic/svi_geo"
        )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", src])

**Authentication and permissions.**

- **Colab:** the next cell calls `google.colab.auth.authenticate_user()`.
- **Local Jupyter:** Application Default Credentials, set up once with
  `gcloud auth application-default login`.
- **Optional workstation helpers** (only if plain ADC fails, e.g. Vertex AI returns 401 on
  some managed workstations): `SVI_USE_GCLOUD_TOKEN=1` mints short-lived user tokens with
  `gcloud auth print-access-token`, and `SVI_ECP_PROXY_URL=http://localhost:<port>` routes
  Vertex AI calls through a local enterprise-certificate (mTLS) proxy.
- **Parameters:** set `PROJECT_ID` (your billing project) and `GCS_BUCKET` (the frame bucket
  linked to your Imagery Insights dataset) in the parameters cell or as environment
  variables. The notebook stops if either is missing; nothing is guessed.
- **IAM roles you need** (this notebook changes no IAM settings): `roles/bigquery.jobUser` on
  `PROJECT_ID`, `roles/bigquery.dataViewer` on the linked dataset,
  `roles/storage.objectViewer` on the frame bucket and `roles/aiplatform.user` for Gemini.

In [ ]:
# Colab: authenticate the notebook user (Application Default Credentials elsewhere).
try:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()
except ImportError:
    pass

In [ ]:
import os

# ---- Parameters -------------------------------------------------------------------------
PROJECT_ID = os.environ.get("PROJECT_ID", "YOUR_PROJECT_ID")  # billing project (required)
DATASET_ID = "imagery_insights___us"  # Imagery Insights dataset linked into PROJECT_ID
GCS_BUCKET = os.environ.get("GCS_BUCKET", "")  # frame bucket linked to DATASET_ID (required)
MODEL = "gemini-3.5-flash"
LOCATION = "global"
MAX_GEMINI_CALLS = 500  # hard cap on Gemini requests for this notebook; None = unlimited
CONCURRENCY = 16  # parallel Gemini requests

LAT, LNG = 28.05047, -81.96015  # target house (roughly the facade centre)
RADIUS_M = SEARCH_RADIUS_M = 80.0
MAX_VIEWS = 12  # Gemini calls
HOUSE_WIDTH_M = 14.0  # crop covers about this width at the house

In [ ]:
from google.cloud import storage

from svi_geo import auth, config, data, images, rosette
from svi_geo import gemini_client as gc

# Fails fast (ConfigError) if PROJECT_ID or GCS_BUCKET is missing; no silent fallbacks.
settings = config.resolve_settings(PROJECT_ID, GCS_BUCKET, env=dict(os.environ), dataset=DATASET_ID)
PROJECT_ID, BUCKET = settings.project, settings.bucket
print("project:", PROJECT_ID, "| frames bucket:", BUCKET)

creds = auth.get_credentials()  # None -> Application Default Credentials
TABLE = data.pano_table(PROJECT_ID, DATASET_ID)  # pano_observations_latest
bq = data.QueryRunner(
    data.make_bigquery_client(PROJECT_ID, creds),
    allowed_tables=data.pano_tables(PROJECT_ID, DATASET_ID),  # pano views only
    cache_dir=data.DEFAULT_QUERY_CACHE,
)
fetcher = images.GcsImageFetcher(storage.Client(project=PROJECT_ID, credentials=creds))
runner = gc.GeminiRunner(
    gc.VertexGeminiBackend(gc.make_vertex_client(PROJECT_ID, LOCATION, creds), model=MODEL),
    max_calls=MAX_GEMINI_CALLS,
    concurrency=CONCURRENCY,
)
intr = rosette.load_intrinsics()  # self-calibrated 7-camera rosette model shipped with svi_geo
print("intrinsics fitted:", intr.fitted)

In [ ]:
import math

import numpy as np
import pandas as pd

from svi_geo import entities as ent
from svi_geo import geo, schemas, sequence

In [ ]:
# Metadata only (no gcs_uri column: selecting it scans ~1.9 GB); the frame URI is derived
# from snapshot_id + observation_id. Parameterised SQL, dry-run first, 2 GB cap.
PANO_META_SQL = data.pano_meta_sql(TABLE)
params = data.pano_meta_params(lat=LAT, lng=LNG, radius_m=RADIUS_M)
print(f"dry run: {bq.dry_run(PANO_META_SQL, params) / 1e9:.2f} GB")
frames = data.normalize_frames(bq.run(PANO_META_SQL, params))
frames["gcs_uri"] = [
    data.gcs_uri_for(BUCKET, s, o)
    for s, o in zip(frames["snapshot_id"], frames["observation_id"], strict=True)
]
panos = sequence.build_sequences(data.panos_from_frames(frames))
panos["travel_deg"] = sequence.travel_bearing(panos)
print(f"{len(frames)} frames, {len(panos)} panos, {panos.seq_id.nunique()} drive sequences")
print("measured pano spacing (m):", sequence.spacing_stats(panos))

## Rank candidate views in code

In [ ]:
cands = []
for _pid, g in frames.groupby("pano_id"):
    ranked_cameras = rosette.rank_cameras_for_target(g.to_dict("records"), LAT, LNG, intr=intr)
    if not ranked_cameras:
        continue

    # Try top 2 ranked cameras
    for row in ranked_cameras[:2]:
        pose = row["camera_pose"]
        dist = float(geo.haversine_m(pose["latitude"], pose["longitude"], LAT, LNG))
        bearing = float(geo.bearing_deg(pose["latitude"], pose["longitude"], LAT, LNG))
        heading = pose["heading"] + intr.cam_rot_delta_deg.get(int(row["cam_k"]), (0.0,))[0]
        off_axis = abs(float(geo.angdiff(heading, bearing)))
        safe_max_hfov = 2 * (48.9 - off_axis)

        target_hfov = 2 * math.degrees(math.atan(HOUSE_WIDTH_M / 2 / dist)) if dist > 0 else 50.0
        hfov = float(np.clip(target_hfov, 25.0, 50.0))

        # verified FOV logic
        if hfov <= safe_max_hfov and 5.0 <= dist <= SEARCH_RADIUS_M:
            cands.append(
                {
                    **row,
                    "dist_m": dist,
                    "bearing": bearing,
                    "off_axis_deg": off_axis,
                    "hfov": hfov,
                    "score": dist / 10.0 + off_axis / 10.0,
                }
            )
            break  # Found valid camera for this pano
cands = pd.DataFrame(cands).sort_values("score").head(MAX_VIEWS).reset_index(drop=True)
print(f"{frames.pano_id.nunique()} panos in radius -> {len(cands)} views selected")
cands[["pano_id", "cam_k", "dist_m", "off_axis_deg", "score"]]

In [ ]:
def house_crop(row):
    img = images.decode(fetcher.fetch(row["gcs_uri"]))
    target_hfov = 2 * math.degrees(math.atan(HOUSE_WIDTH_M / 2 / row["dist_m"]))

    # Fallback to cap HFOV to stay within KB4 half-FOV limit without black borders
    safe_max_hfov = max(5.0, 2 * (34.9 - row["off_axis_deg"]))
    hfov = float(np.clip(target_hfov, 5.0, safe_max_hfov))

    view = rosette.PerspectiveView(row["bearing"], 8.0, hfov, 1024, 768)
    return rosette.render_perspective(img, intr, row["camera_pose"], view, int(row["cam_k"]))


crops = [house_crop(r) for r in cands.to_dict("records")]
PROMPT = (
    "The image is centred on the bearing of a target house. Is a house visible at the CENTRE? "
    "If yes, box it ([ymin, xmin, ymax, xmax], 0-1000), and report occlusion, the visible "
    "fraction of its facade, stories, exterior material and roof type, with your confidence."
)
print(f"planned: {len(crops)} calls, est. ${gc.estimate_cost(len(crops)):.2f}")
# The API enforces the HouseView schema (JSON mode); code rejects degenerate or out-of-range
# boxes and re-asks once.
CROP_H, CROP_W = crops[0].shape[:2]
views = await runner.ask_many(
    [([PROMPT, c], schemas.HouseView) for c in crops],
    validator=lambda v: v.validate_in_image(CROP_W, CROP_H),
)
print(runner.cost.summary())
runner.check()  # raises if any request failed, with the first error messages
assert any(v is not None and v.house_visible for v in views), "no candidate shows the house"

In [ ]:
house_id = ent.entity_id_for("HOUSE", LAT, LNG)
res = cands.assign(
    visible=[v.house_visible if v else None for v in views],
    occlusion=[v.occlusion.value if v else None for v in views],
    facade_fraction=[v.facade_visible_fraction if v else None for v in views],
    confidence=[v.confidence if v else None for v in views],
)
# a box far from the centre means the model looked at a different building
res["centred"] = [
    bool(v and v.box_2d and abs((v.box_2d[1] + v.box_2d[3]) / 2 - 500) < 250) for v in views
]
ok = [v for v, c in zip(views, res.centred, strict=True) if v and v.house_visible and c]


def value(x):
    return getattr(x, "value", x)


attrs = {
    k: ent.fuse_attribute((value(getattr(v, k)), v.confidence) for v in ok)
    for k in ("stories", "exterior_material", "roof_type")
}
print("house_id:", house_id)
print("fused attributes (value, weight share):", attrs)
res[
    [
        "pano_id",
        "cam_k",
        "dist_m",
        "visible",
        "centred",
        "occlusion",
        "facade_fraction",
        "confidence",
    ]
]

In [ ]:
import matplotlib.pyplot as plt

best = (
    res[res.visible.fillna(False) & res.centred]
    .sort_values(["facade_fraction", "confidence"], ascending=False)
    .head(4)
)
fig, axes = plt.subplots(1, max(1, len(best)), figsize=(5 * max(1, len(best)), 4))
for ax, (i, r) in zip(np.atleast_1d(axes), best.iterrows(), strict=False):
    ax.imshow(crops[i][..., ::-1])
    ax.set_title(f"{r.dist_m:.0f} m, facade {r.facade_fraction:.2f}")
    ax.axis("off")
plt.show()
print(runner.cost.summary())